## 1. 생활계 폐기물 발생량 및 처리 현황 데이터 병합

In [1]:
import pandas as pd
from pathlib import Path

# =========================
# 1. 폴더 경로 설정
# =========================
base_path = Path("./data")
raw_path = base_path / "raw"
processed_path = base_path / "processed"
final_path = base_path / "final"

processed_path.mkdir(parents=True, exist_ok=True)
final_path.mkdir(parents=True, exist_ok=True)


In [2]:
# =========================
# 2. 파일명 설정
# =========================
file_old = raw_path / "생활계폐기물발생량및처리현황(2019년이전).csv"
file_new = raw_path / "생활계폐기물발생량및처리현황(2020년이후).csv"


In [4]:
# =========================
# 3. CSV 읽기 함수
# =========================
def read_waste_file(file_path):
    df = pd.read_csv(file_path, header=[0, 1], encoding="utf-8")
    
    new_cols = []
    for col1, col2 in df.columns:
        col1 = str(col1).strip() if pd.notna(col1) else ""
        col2 = str(col2).strip() if pd.notna(col2) else ""

        if "Unnamed" in col1:
            col1 = ""
        if "Unnamed" in col2:
            col2 = ""

        if col1 and col2:
            new_col = f"{col1}_{col2}"
        elif col1:
            new_col = col1
        else:
            new_col = col2

        new_cols.append(new_col)

    df.columns = [c.replace("\n", "").replace("  ", " ").strip() for c in new_cols]
    return df

In [5]:
# =========================
# 4. 컬럼명 표준화
# =========================
def standardize_columns(df):
    col_map = {}

    for col in df.columns:
        c = col.replace(" ", "")

        if "구분별(1)" in c:
            col_map[col] = "대분류"
        elif "구분별(2)" in c:
            col_map[col] = "자치구"
        elif "시점" in c:
            col_map[col] = "연도"
        elif "발생량" in c and "소계" in c:
            col_map[col] = "발생량"
        elif "재활용" in c and "소계" in c:
            col_map[col] = "재활용_합계"
        elif c == "재활용_재활용" or ("재활용" in c and "음식물" not in c and "소계" not in c):
            col_map[col] = "재활용"
        elif "음식물" in c:
            col_map[col] = "음식물"
        elif "소각" in c:
            col_map[col] = "소각"
        elif "매립" in c:
            col_map[col] = "매립"
        elif "기타" in c:
            col_map[col] = "기타"

    return df.rename(columns=col_map)


In [6]:
# =========================
# 5. 정리 함수
# =========================
def clean_waste_df(df):
    df = standardize_columns(df)

    needed_cols = ["대분류", "자치구", "연도", "발생량", "재활용_합계", "재활용", "음식물", "소각", "매립", "기타"]
    existing_cols = [c for c in needed_cols if c in df.columns]
    df = df[existing_cols].copy()

    for col in ["대분류", "자치구"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    if "자치구" in df.columns:
        df = df[~df["자치구"].isin(["처리비율", "소계", "합계", "총계"])]

    df = df[df["자치구"].notna()]

    if "연도" in df.columns:
        df["연도"] = pd.to_numeric(df["연도"], errors="coerce")

    value_cols = ["발생량", "재활용_합계", "재활용", "음식물", "소각", "매립", "기타"]
    for col in value_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .str.replace("-", "", regex=False)
                .str.strip()
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    gu_list = [
        "종로구", "중구", "용산구", "성동구", "광진구",
        "동대문구", "중랑구", "성북구", "강북구", "도봉구",
        "노원구", "은평구", "서대문구", "마포구", "양천구",
        "강서구", "구로구", "금천구", "영등포구", "동작구",
        "관악구", "서초구", "강남구", "송파구", "강동구"
    ]
    df = df[df["자치구"].isin(gu_list)]

    return df.sort_values(["자치구", "연도"]).reset_index(drop=True)


In [40]:
# =========================
# 6. 파일 읽기 + 정리
# =========================
df_old_raw = read_waste_file(file_old)
df_new_raw = read_waste_file(file_new)

df_old = clean_waste_df(df_old_raw)
df_new = clean_waste_df(df_new_raw)

# 중간 저장
df_old.to_csv(processed_path / "생활폐기물_발생및처리_2000_2019_clean.csv", index=False, encoding="utf-8-sig")
df_new.to_csv(processed_path / "생활폐기물_발생및처리_2020_2024_clean.csv", index=False, encoding="utf-8-sig")


In [41]:
# =========================
# 7. 합치기
# =========================
df_all = pd.concat([df_old, df_new], ignore_index=True).drop_duplicates()
df_all = df_all.sort_values(["자치구", "연도"]).reset_index(drop=True)

df_all.to_csv(final_path / "생활폐기물_발생및처리_2000_2024_master.csv", index=False, encoding="utf-8-sig")

print("전처리 완료")
print(df_all.head())
print(df_all.info())

전처리 완료
  대분류  자치구    연도    발생량  재활용_합계    재활용    음식물     소각     매립  기타
0   계  강남구  2000  794.1   377.0  207.0  170.0   32.3  384.8 NaN
1   계  강남구  2001  799.0   510.7  242.1  268.6  158.7  129.6 NaN
2   계  강남구  2002  861.3   567.9  295.0  272.9  162.8  130.6 NaN
3   계  강남구  2003  874.9   556.8  267.3  289.5  169.5  148.6 NaN
4   계  강남구  2004  835.0   541.4  242.4  299.0  168.0  125.6 NaN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 625 entries, 0 to 624
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   대분류     625 non-null    object 
 1   자치구     625 non-null    object 
 2   연도      625 non-null    int64  
 3   발생량     625 non-null    float64
 4   재활용_합계  625 non-null    float64
 5   재활용     625 non-null    float64
 6   음식물     625 non-null    float64
 7   소각      614 non-null    float64
 8   매립      625 non-null    float64
 9   기타      130 non-null    float64
dtypes: float64(7), int64(1), object(2)
memory usage: 49.0+ 

In [42]:
print(df_all.shape)
print(df_all["자치구"].nunique())
print(df_all["연도"].min(), df_all["연도"].max())
print(df_all.head())

(625, 10)
25
2000 2024
  대분류  자치구    연도    발생량  재활용_합계    재활용    음식물     소각     매립  기타
0   계  강남구  2000  794.1   377.0  207.0  170.0   32.3  384.8 NaN
1   계  강남구  2001  799.0   510.7  242.1  268.6  158.7  129.6 NaN
2   계  강남구  2002  861.3   567.9  295.0  272.9  162.8  130.6 NaN
3   계  강남구  2003  874.9   556.8  267.3  289.5  169.5  148.6 NaN
4   계  강남구  2004  835.0   541.4  242.4  299.0  168.0  125.6 NaN


## 2. 주민 1인당 생활 폐기물 쓰레기 데이터 병합

In [ ]:
import pandas as pd
from pathlib import Path
# =========================
# 2. 파일 불러오기
# =========================

file_master = final_path / "생활폐기물_발생및처리_2000_2024_master.csv"


In [15]:

# =========================
# 3. 1인당 생활폐기물 파일 읽기
# =========================
df_pc = pd.read_csv("../data/raw/주민1인당생활폐기물(쓰레기)배출량.csv", encoding="utf-8")

print(df_pc.head())
print(df_pc.columns.tolist())

  자치구별(1)    시점  주민 1인당 생활폐기물(쓰레기) 배출량 (㎏/인 일)  생활계폐기물 배출량 (톤/일)   주민수 (명)
0       계  2009                           1.08           11336.8  10464051
1       계  2010                           0.95           10020.4  10575447
2       계  2011                           0.90            9440.1  10528774
3       계  2012                           0.88            9189.3  10442426
4       계  2013                           0.82            8559.0  10388055
['자치구별(1)', '시점', '주민 1인당 생활폐기물(쓰레기) 배출량 (㎏/인 일)', '생활계폐기물 배출량 (톤/일)', '주민수 (명)']


In [17]:
# =========================
# 4. 컬럼명 변경
# =========================
df_pc = df_pc.rename(columns={
    "자치구별(1)": "자치구",
    "시점": "연도",
    "주민 1인당 생활폐기물(쓰레기) 배출량 (㎏/인 일)": "주민1인당_생활폐기물_kg_일",
    "생활계폐기물 배출량 (톤/일)": "생활계폐기물_톤_일",
    "주민수 (명)": "주민수"
})

# =========================
# 5. 불필요 행 제거
# =========================
df_pc["자치구"] = df_pc["자치구"].astype(str).str.strip()
df_pc = df_pc[df_pc["자치구"] != "계"].copy()

# =========================
# 6. 숫자형 변환
# =========================
df_pc["연도"] = pd.to_numeric(df_pc["연도"], errors="coerce")
df_pc["주민1인당_생활폐기물_kg_일"] = pd.to_numeric(df_pc["주민1인당_생활폐기물_kg_일"], errors="coerce")
df_pc["생활계폐기물_톤_일"] = pd.to_numeric(df_pc["생활계폐기물_톤_일"], errors="coerce")
df_pc["주민수"] = pd.to_numeric(df_pc["주민수"], errors="coerce")

# =========================
# 7. 서울 25개 자치구만 남기기
# =========================
gu_list = [
    "종로구", "중구", "용산구", "성동구", "광진구",
    "동대문구", "중랑구", "성북구", "강북구", "도봉구",
    "노원구", "은평구", "서대문구", "마포구", "양천구",
    "강서구", "구로구", "금천구", "영등포구", "동작구",
    "관악구", "서초구", "강남구", "송파구", "강동구"
]
df_pc = df_pc[df_pc["자치구"].isin(gu_list)].copy()

# =========================
# 8. 정렬 및 저장
# =========================
df_pc = df_pc.sort_values(["자치구", "연도"]).reset_index(drop=True)
df_pc.to_csv(processed_path / "주민1인당_생활폐기물_clean.csv", index=False, encoding="utf-8-sig")

print(df_pc.head())
print(df_pc.info())

   자치구    연도  주민1인당_생활폐기물_kg_일  생활계폐기물_톤_일     주민수
0  강남구  2009              1.72       978.7  569499
1  강남구  2010              1.72       990.5  577070
2  강남구  2011              1.78      1017.3  573003
3  강남구  2012              1.73       984.9  569997
4  강남구  2013              1.31       745.2  569152
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   자치구               400 non-null    object 
 1   연도                400 non-null    int64  
 2   주민1인당_생활폐기물_kg_일  400 non-null    float64
 3   생활계폐기물_톤_일        400 non-null    float64
 4   주민수               400 non-null    int64  
dtypes: float64(2), int64(2), object(1)
memory usage: 15.8+ KB
None


In [20]:
# =========================
# 9. 기존 master 불러오기
# =========================

file_master = "../data/final/생활폐기물_발생및처리_2000_2024_master.csv"
df_master = pd.read_csv(file_master, encoding="utf-8-sig")

print(df_master.head())
print(df_master.columns.tolist())

# =========================
# 10. merge
# =========================
df_merged = df_master.merge(
    df_pc,
    on=["자치구", "연도"],
    how="left"
)

# =========================
# 11. 저장
# =========================
df_merged.to_csv("../data/final/생활폐기물_발생및처리_1인당포함_master.csv", index=False, encoding="utf-8-sig")

print(df_merged.head())
print(df_merged.info())
print(df_merged[["자치구", "연도", "발생량", "주민1인당_생활폐기물_kg_일", "주민수"]].head(10))

  대분류  자치구    연도    발생량  재활용_합계    재활용    음식물     소각     매립  기타
0   계  강남구  2000  794.1   377.0  207.0  170.0   32.3  384.8 NaN
1   계  강남구  2001  799.0   510.7  242.1  268.6  158.7  129.6 NaN
2   계  강남구  2002  861.3   567.9  295.0  272.9  162.8  130.6 NaN
3   계  강남구  2003  874.9   556.8  267.3  289.5  169.5  148.6 NaN
4   계  강남구  2004  835.0   541.4  242.4  299.0  168.0  125.6 NaN
['대분류', '자치구', '연도', '발생량', '재활용_합계', '재활용', '음식물', '소각', '매립', '기타']
  대분류  자치구    연도    발생량  재활용_합계    재활용    음식물     소각     매립  기타  \
0   계  강남구  2000  794.1   377.0  207.0  170.0   32.3  384.8 NaN   
1   계  강남구  2001  799.0   510.7  242.1  268.6  158.7  129.6 NaN   
2   계  강남구  2002  861.3   567.9  295.0  272.9  162.8  130.6 NaN   
3   계  강남구  2003  874.9   556.8  267.3  289.5  169.5  148.6 NaN   
4   계  강남구  2004  835.0   541.4  242.4  299.0  168.0  125.6 NaN   

   주민1인당_생활폐기물_kg_일  생활계폐기물_톤_일  주민수  
0               NaN         NaN  NaN  
1               NaN         NaN  NaN  
2               NaN       

In [46]:
print(df_merged["주민1인당_생활폐기물_kg_일"].isna().sum())
print(df_merged["주민수"].isna().sum())
print(df_merged["연도"].min(), df_merged["연도"].max())
print(df_merged["자치구"].nunique())

225
225
2000 2024
25


## 3. 폐기물재활용현황 데이터 구축

In [22]:
import pandas as pd
from pathlib import Path



# =========================
# 파일명
# =========================
file_recycle_old = "../data/raw/폐기물재활용현황(2019년이전).csv"
file_recycle_new = "../data/raw/폐기물재활용현황(2020년이후).csv"
file_master = "../data/final/생활폐기물_발생및처리_1인당포함_master.csv"

# =========================
#  CSV 읽기
# =========================
def read_recycle_file(file_path, encoding="utf-8"):
    df = pd.read_csv(file_path, header=[0, 1], encoding=encoding)

    new_cols = []
    for col1, col2 in df.columns:
        col1 = str(col1).strip() if pd.notna(col1) else ""
        col2 = str(col2).strip() if pd.notna(col2) else ""

        if "Unnamed" in col1:
            col1 = ""
        if "Unnamed" in col2:
            col2 = ""

        if col1 and col2:
            new_col = f"{col1}_{col2}"
        elif col1:
            new_col = col1
        else:
            new_col = col2

        new_cols.append(new_col)

    df.columns = [c.replace("\n", "").replace("  ", " ").strip() for c in new_cols]
    return df


try:
    df_re_old_raw = read_recycle_file(file_recycle_old, encoding="utf-8")
    df_re_new_raw = read_recycle_file(file_recycle_new, encoding="utf-8")
except:
    df_re_old_raw = read_recycle_file(file_recycle_old, encoding="cp949")
    df_re_new_raw = read_recycle_file(file_recycle_new, encoding="cp949")

print(df_re_old_raw.head())
print(df_re_old_raw.columns.tolist())

  자치구별(1)_자치구별(1) 자치구별(2)_자치구별(2)  시점_시점 합계_발생량  합계_재활용량 생활폐기물_발생량  \
0             서울시              소계   2000      -  18179.0         -   
1             서울시              소계   2001      -  26462.0         -   
2             서울시              소계   2002      -  28202.0         -   
3             서울시              소계   2003      -  31919.0         -   
4             서울시              소계   2004      -  31336.0         -   

   생활폐기물_재활용량 음식물폐기물_발생량 음식물폐기물_재활용량 사업장배출 시설계폐기물_발생량 사업장배출 시설계폐기물_재활용량  \
0      5048.0          -           -                -             293.0   
1      5682.0          -           -                -             329.0   
2      5852.0          -           -                -             496.0   
3      6176.0          -           -                -             689.0   
4      4020.0          -      2406.0                -             622.0   

  건설폐기물_발생량 건설폐기물_재활용량 지정폐기물_발생량 지정폐기물_재활용량  
0         -    12838.0         -          -  
1         -    20451.0         -    

In [23]:
def standardize_recycle_columns(df):
    col_map = {}

    for col in df.columns:
        c = col.replace(" ", "").strip()

        # 기본 키
        if "자치구별(1)" in c:
            col_map[col] = "광역시도"
        elif "자치구별(2)" in c:
            col_map[col] = "자치구"
        elif "시점" in c:
            col_map[col] = "연도"

        # 재활용률
        elif "재활용률" in c:
            col_map[col] = "재활용률"

        # 합계
        elif "합계" in c and "발생량" in c:
            col_map[col] = "재활용총량_발생량"
        elif "합계" in c and ("재활용량" in c or "재활용" in c):
            col_map[col] = "재활용총량_재활용량"

        # 생활폐기물 / 생활계 폐기물
        elif ("생활폐기물" in c or "생활계폐기물" in c) and "발생량" in c:
            col_map[col] = "생활폐기물재활용통계_발생량"
        elif ("생활폐기물" in c or "생활계폐기물" in c) and ("재활용량" in c or "재활용" in c):
            col_map[col] = "생활폐기물재활용통계_재활용량"

        # 음식물폐기물 (구버전에만 있음)
        elif "음식물폐기물" in c and "발생량" in c:
            col_map[col] = "음식물폐기물_발생량"
        elif "음식물폐기물" in c and ("재활용량" in c or "재활용" in c):
            col_map[col] = "음식물폐기물_재활용량"

        # 사업장배출시설계폐기물
        elif "사업장배출시설계폐기물" in c and "발생량" in c:
            col_map[col] = "사업장배출시설계_발생량"
        elif "사업장배출시설계폐기물" in c and ("재활용량" in c or "재활용" in c):
            col_map[col] = "사업장배출시설계_재활용량"

        # 건설폐기물
        elif "건설폐기물" in c and "발생량" in c:
            col_map[col] = "건설폐기물_발생량"
        elif "건설폐기물" in c and ("재활용량" in c or "재활용" in c):
            col_map[col] = "건설폐기물_재활용량"

        # 지정폐기물
        elif "지정폐기물" in c and "발생량" in c:
            col_map[col] = "지정폐기물_발생량"
        elif "지정폐기물" in c and ("재활용량" in c or "재활용" in c):
            col_map[col] = "지정폐기물_재활용량"

    return df.rename(columns=col_map)

In [24]:
def clean_recycle_df(df):
    df = standardize_recycle_columns(df)

    needed_cols = [
        "광역시도", "자치구", "연도", "재활용률",
        "재활용총량_발생량", "재활용총량_재활용량",
        "생활폐기물재활용통계_발생량", "생활폐기물재활용통계_재활용량",
        "음식물폐기물_발생량", "음식물폐기물_재활용량",
        "사업장배출시설계_발생량", "사업장배출시설계_재활용량",
        "건설폐기물_발생량", "건설폐기물_재활용량",
        "지정폐기물_발생량", "지정폐기물_재활용량"
    ]
    existing_cols = [c for c in needed_cols if c in df.columns]
    df = df[existing_cols].copy()

    for col in ["광역시도", "자치구"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    if "광역시도" in df.columns:
        df = df[df["광역시도"].isin(["서울시", "서울특별시"])]

    if "자치구" in df.columns:
        df = df[df["자치구"] != "소계"]

    gu_list = [
        "종로구", "중구", "용산구", "성동구", "광진구",
        "동대문구", "중랑구", "성북구", "강북구", "도봉구",
        "노원구", "은평구", "서대문구", "마포구", "양천구",
        "강서구", "구로구", "금천구", "영등포구", "동작구",
        "관악구", "서초구", "강남구", "송파구", "강동구"
    ]
    if "자치구" in df.columns:
        df = df[df["자치구"].isin(gu_list)].copy()

    if "연도" not in df.columns:
        raise ValueError(f"'연도' 컬럼이 없습니다. 현재 컬럼: {df.columns.tolist()}")

    df["연도"] = pd.to_numeric(df["연도"], errors="coerce")

    value_cols = [c for c in df.columns if c not in ["광역시도", "자치구", "연도"]]
    for col in value_cols:
        df[col] = (
            df[col].astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("-", "", regex=False)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.sort_values(["자치구", "연도"]).reset_index(drop=True)

In [25]:
df_re_old = clean_recycle_df(df_re_old_raw)
df_re_new = clean_recycle_df(df_re_new_raw)

df_re_old.to_csv(processed_path / "폐기물재활용현황_2000_2019_clean.csv", index=False, encoding="utf-8-sig")
df_re_new.to_csv(processed_path / "폐기물재활용현황_2020_2024_clean.csv", index=False, encoding="utf-8-sig")

df_re_all = pd.concat([df_re_old, df_re_new], ignore_index=True).drop_duplicates()
df_re_all = df_re_all.sort_values(["자치구", "연도"]).reset_index(drop=True)

df_re_all.to_csv(processed_path / "폐기물재활용현황_2000_2024_master.csv", index=False, encoding="utf-8-sig")

print(df_re_all.head())
print(df_re_all.info())

  광역시도  자치구    연도  재활용총량_발생량  재활용총량_재활용량  생활폐기물재활용통계_발생량  생활폐기물재활용통계_재활용량  \
0  서울시  강남구  2000        NaN       594.0             NaN            339.0   
1  서울시  강남구  2001        NaN       574.0             NaN            511.0   
2  서울시  강남구  2002        NaN       599.0             NaN            568.0   
3  서울시  강남구  2003        NaN      3345.0             NaN            557.0   
4  서울시  강남구  2004        NaN      2832.0             NaN            242.0   

   음식물폐기물_발생량  음식물폐기물_재활용량  사업장배출시설계_발생량  사업장배출시설계_재활용량  건설폐기물_발생량  \
0         NaN          NaN           NaN            NaN        NaN   
1         NaN          NaN           NaN            NaN        NaN   
2         NaN          NaN           NaN           32.0        NaN   
3         NaN          NaN           NaN           32.0        NaN   
4         NaN        299.0           NaN           28.0        NaN   

   건설폐기물_재활용량  지정폐기물_발생량  지정폐기물_재활용량  재활용률  
0       255.0        NaN         NaN   NaN  
1        63.0        NaN  

In [26]:
df_re_old = clean_recycle_df(df_re_old_raw)
df_re_new = clean_recycle_df(df_re_new_raw)

print(df_re_old.head())
print(df_re_new.head())
print(df_re_old.columns.tolist())
print(df_re_new.columns.tolist())

  광역시도  자치구    연도  재활용총량_발생량  재활용총량_재활용량  생활폐기물재활용통계_발생량  생활폐기물재활용통계_재활용량  \
0  서울시  강남구  2000        NaN       594.0             NaN            339.0   
1  서울시  강남구  2001        NaN       574.0             NaN            511.0   
2  서울시  강남구  2002        NaN       599.0             NaN            568.0   
3  서울시  강남구  2003        NaN      3345.0             NaN            557.0   
4  서울시  강남구  2004        NaN      2832.0             NaN            242.0   

   음식물폐기물_발생량  음식물폐기물_재활용량  사업장배출시설계_발생량  사업장배출시설계_재활용량  건설폐기물_발생량  \
0         NaN          NaN           NaN            NaN        NaN   
1         NaN          NaN           NaN            NaN        NaN   
2         NaN          NaN           NaN           32.0        NaN   
3         NaN          NaN           NaN           32.0        NaN   
4         NaN        299.0           NaN           28.0        NaN   

   건설폐기물_재활용량  지정폐기물_발생량  지정폐기물_재활용량  
0       255.0        NaN         NaN  
1        63.0        NaN         NaN  

In [27]:
df_re_all = pd.concat([df_re_old, df_re_new], ignore_index=True).drop_duplicates()
df_re_all = df_re_all.sort_values(["자치구", "연도"]).reset_index(drop=True)

df_re_all.to_csv(processed_path / "폐기물재활용현황_2000_2024_master.csv", index=False, encoding="utf-8-sig")

df_master = pd.read_csv(file_master, encoding="utf-8-sig")

df_merged2 = df_master.merge(
    df_re_all,
    on=["자치구", "연도"],
    how="left"
)

df_merged2.to_csv(final_path / "생활폐기물_발생및처리_1인당_재활용포함_master.csv", index=False, encoding="utf-8-sig")

print(df_merged2.head())
print(df_merged2.info())

  대분류  자치구    연도    발생량  재활용_합계    재활용    음식물     소각     매립  기타  ...  \
0   계  강남구  2000  794.1   377.0  207.0  170.0   32.3  384.8 NaN  ...   
1   계  강남구  2001  799.0   510.7  242.1  268.6  158.7  129.6 NaN  ...   
2   계  강남구  2002  861.3   567.9  295.0  272.9  162.8  130.6 NaN  ...   
3   계  강남구  2003  874.9   556.8  267.3  289.5  169.5  148.6 NaN  ...   
4   계  강남구  2004  835.0   541.4  242.4  299.0  168.0  125.6 NaN  ...   

   생활폐기물재활용통계_재활용량  음식물폐기물_발생량  음식물폐기물_재활용량 사업장배출시설계_발생량  사업장배출시설계_재활용량  \
0            339.0         NaN          NaN          NaN            NaN   
1            511.0         NaN          NaN          NaN            NaN   
2            568.0         NaN          NaN          NaN           32.0   
3            557.0         NaN          NaN          NaN           32.0   
4            242.0         NaN        299.0          NaN           28.0   

   건설폐기물_발생량  건설폐기물_재활용량  지정폐기물_발생량  지정폐기물_재활용량  재활용률  
0        NaN       255.0        NaN         NaN   NaN  
1   

In [28]:
df_re_all = pd.concat([df_re_old, df_re_new], ignore_index=True).drop_duplicates()
df_re_all = df_re_all.sort_values(["자치구", "연도"]).reset_index(drop=True)

df_re_all.to_csv(processed_path / "폐기물재활용현황_2000_2024_master.csv", index=False, encoding="utf-8-sig")

df_master = pd.read_csv(file_master, encoding="utf-8-sig")

df_merged2 = df_master.merge(
    df_re_all,
    on=["자치구", "연도"],
    how="left"
)

df_merged2.to_csv(final_path / "생활폐기물_발생및처리_1인당_재활용포함_master.csv", index=False, encoding="utf-8-sig")

print(df_merged2.head())
print(df_merged2.info())

  대분류  자치구    연도    발생량  재활용_합계    재활용    음식물     소각     매립  기타  ...  \
0   계  강남구  2000  794.1   377.0  207.0  170.0   32.3  384.8 NaN  ...   
1   계  강남구  2001  799.0   510.7  242.1  268.6  158.7  129.6 NaN  ...   
2   계  강남구  2002  861.3   567.9  295.0  272.9  162.8  130.6 NaN  ...   
3   계  강남구  2003  874.9   556.8  267.3  289.5  169.5  148.6 NaN  ...   
4   계  강남구  2004  835.0   541.4  242.4  299.0  168.0  125.6 NaN  ...   

   생활폐기물재활용통계_재활용량  음식물폐기물_발생량  음식물폐기물_재활용량 사업장배출시설계_발생량  사업장배출시설계_재활용량  \
0            339.0         NaN          NaN          NaN            NaN   
1            511.0         NaN          NaN          NaN            NaN   
2            568.0         NaN          NaN          NaN           32.0   
3            557.0         NaN          NaN          NaN           32.0   
4            242.0         NaN        299.0          NaN           28.0   

   건설폐기물_발생량  건설폐기물_재활용량  지정폐기물_발생량  지정폐기물_재활용량  재활용률  
0        NaN       255.0        NaN         NaN   NaN  
1   

## 음식물류 폐기량 발생원별 발생량 데이터 구축

In [29]:
import pandas as pd
from pathlib import Path


# =========================
# 2. 파일명
# =========================
file_food_source = "../data/raw/음식물류폐기물발생원별발생량.csv"
file_master = "../data/final/생활폐기물_발생및처리_1인당_재활용포함_master.csv"

In [30]:
def read_food_source_file(file_path, encoding="utf-8"):
    df = pd.read_csv(file_path, header=[0, 1, 2], encoding=encoding)

    new_cols = []
    for c1, c2, c3 in df.columns:
        c1 = str(c1).strip() if pd.notna(c1) else ""
        c2 = str(c2).strip() if pd.notna(c2) else ""
        c3 = str(c3).strip() if pd.notna(c3) else ""

        if "Unnamed" in c1:
            c1 = ""
        if "Unnamed" in c2:
            c2 = ""
        if "Unnamed" in c3:
            c3 = ""

        parts = [x for x in [c1, c2, c3] if x]
        new_cols.append("_".join(parts))

    df.columns = [c.replace("\n", "").replace("  ", " ").strip() for c in new_cols]
    return df

try:
    df_food_raw = read_food_source_file(file_food_source, encoding="utf-8")
except:
    df_food_raw = read_food_source_file(file_food_source, encoding="cp949")

print(df_food_raw.head())
print(df_food_raw.columns.tolist())

  자치구별(1)_자치구별(1)_자치구별(1) 자치구별(2)_자치구별(2)_자치구별(2)  시점_시점_시점  발생량_소계_소계  \
0                     서울시                      소계      2014     3181.1   
1                     서울시                      소계      2015     3165.8   
2                     서울시                      소계      2016     3075.0   
3                     서울시                      소계      2017     2871.7   
4                     서울시                      소계      2018     2818.9   

   발생량_가정_소계  발생량_가정_단독주택  발생량_가정_공동주택  발생량_가정_소형음식점  발생량_다량배출사업장_소계  \
0     2548.0        980.0        971.8         596.2           633.1   
1     2514.7        983.0        908.0         623.7           651.1   
2     2382.3        903.2        842.2         636.9           692.7   
3     2211.9        812.8        778.4         620.7           659.8   
4     2143.6        780.1        756.3         607.2           675.3   

   발생량_다량배출사업장_집단급식소  발생량_다량배출사업장_음식점(200㎡ 이상) 발생량_다량배출사업장_대규모점포  \
0              168.9                     301.6        

In [31]:
def standardize_food_source_columns(df):
    col_map = {}

    for col in df.columns:
        c = col.replace(" ", "").replace("(", "").replace(")", "").strip()

        # 기본 키
        if "자치구별(1)" in col:
            col_map[col] = "광역시도"
        elif "자치구별(2)" in col:
            col_map[col] = "자치구"
        elif "시점" in col:
            col_map[col] = "연도"

        # 전체
        elif "발생량_소계_소계" in c:
            col_map[col] = "음식물총량"

        # 가정
        elif "발생량_가정_소계" in c:
            col_map[col] = "가정_소계"
        elif "발생량_가정_단독주택" in c:
            col_map[col] = "가정_단독주택"
        elif "발생량_가정_공동주택" in c:
            col_map[col] = "가정_공동주택"
        elif "발생량_가정_소형음식점" in c:
            col_map[col] = "가정_소형음식점"

        # 다량배출사업장
        elif "발생량_다량배출사업장_소계" in c:
            col_map[col] = "다량배출사업장_소계"
        elif "발생량_다량배출사업장_집단급식소" in c:
            col_map[col] = "다량배출사업장_집단급식소"
        elif "발생량_다량배출사업장_음식점200㎡이상" in c or "발생량_다량배출사업장_음식점200㎡이상".replace("㎡","") in c:
            col_map[col] = "다량배출사업장_음식점200이상"
        elif "발생량_다량배출사업장_대규모점포" in c:
            col_map[col] = "다량배출사업장_대규모점포"
        elif "발생량_다량배출사업장_농수산시장" in c:
            col_map[col] = "다량배출사업장_농수산시장"
        elif "발생량_다량배출사업장_관광숙박시설" in c:
            col_map[col] = "다량배출사업장_관광숙박시설"

        # 사업장폐기물 배출자 사업장
        elif "발생량_사업장폐기물배출자사업장_소계" in c:
            col_map[col] = "사업장폐기물배출자_소계"
        elif "발생량_사업장폐기물배출자사업장_집단급식소" in c:
            col_map[col] = "사업장폐기물배출자_집단급식소"
        elif "발생량_사업장폐기물배출자사업장_음식점200㎡이상" in c or "발생량_사업장폐기물배출자사업장_음식점200㎡이상".replace("㎡","") in c:
            col_map[col] = "사업장폐기물배출자_음식점200이상"
        elif "발생량_사업장폐기물배출자사업장_대규모점포" in c:
            col_map[col] = "사업장폐기물배출자_대규모점포"
        elif "발생량_사업장폐기물배출자사업장_농수산시장" in c:
            col_map[col] = "사업장폐기물배출자_농수산시장"
        elif "발생량_사업장폐기물배출자사업장_관광숙박시설" in c:
            col_map[col] = "사업장폐기물배출자_관광숙박시설"

    return df.rename(columns=col_map)

In [32]:
def clean_food_source_df(df):
    df = standardize_food_source_columns(df)

    needed_cols = [
        "광역시도", "자치구", "연도",
        "음식물총량",
        "가정_소계", "가정_단독주택", "가정_공동주택", "가정_소형음식점",
        "다량배출사업장_소계", "다량배출사업장_집단급식소",
        "다량배출사업장_음식점200이상", "다량배출사업장_대규모점포",
        "다량배출사업장_농수산시장", "다량배출사업장_관광숙박시설",
        "사업장폐기물배출자_소계", "사업장폐기물배출자_집단급식소",
        "사업장폐기물배출자_음식점200이상", "사업장폐기물배출자_대규모점포",
        "사업장폐기물배출자_농수산시장", "사업장폐기물배출자_관광숙박시설"
    ]

    existing_cols = [c for c in needed_cols if c in df.columns]
    df = df[existing_cols].copy()

    for col in ["광역시도", "자치구"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    # 서울만
    if "광역시도" in df.columns:
        df = df[df["광역시도"].isin(["서울시", "서울특별시"])]

    # 서울 전체 소계 제거
    if "자치구" in df.columns:
        df = df[df["자치구"] != "소계"]

    gu_list = [
        "종로구", "중구", "용산구", "성동구", "광진구",
        "동대문구", "중랑구", "성북구", "강북구", "도봉구",
        "노원구", "은평구", "서대문구", "마포구", "양천구",
        "강서구", "구로구", "금천구", "영등포구", "동작구",
        "관악구", "서초구", "강남구", "송파구", "강동구"
    ]
    df = df[df["자치구"].isin(gu_list)].copy()

    df["연도"] = pd.to_numeric(df["연도"], errors="coerce")

    value_cols = [c for c in df.columns if c not in ["광역시도", "자치구", "연도"]]
    for col in value_cols:
        df[col] = (
            df[col].astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("-", "", regex=False)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.sort_values(["자치구", "연도"]).reset_index(drop=True)

In [33]:
df_food = clean_food_source_df(df_food_raw)

df_food.to_csv(
    processed_path / "음식물류폐기물_발생원별_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df_food.head())
print(df_food.columns.tolist())
print(df_food.info())

  광역시도  자치구    연도  음식물총량  가정_소계  가정_단독주택  가정_공동주택  가정_소형음식점  다량배출사업장_소계  \
0  서울시  강남구  2014  342.3  190.5     43.2     52.2      95.1       151.8   
1  서울시  강남구  2015  272.2  192.8     47.5     56.9      88.4        79.4   
2  서울시  강남구  2016  287.0  188.9     42.1     52.1      94.7        98.1   
3  서울시  강남구  2017  261.8  180.1     42.2     51.6      86.3        81.7   
4  서울시  강남구  2018  269.5  177.1     38.9     47.6      90.6        92.4   

   다량배출사업장_집단급식소  다량배출사업장_음식점200이상  다량배출사업장_대규모점포  다량배출사업장_농수산시장  \
0           25.0              99.9           11.5            NaN   
1           20.8              44.8            8.7            NaN   
2           22.6              55.9            9.8            NaN   
3           12.0              50.0           12.0            NaN   
4           19.1              51.6           14.7            NaN   

   다량배출사업장_관광숙박시설  사업장폐기물배출자_소계  사업장폐기물배출자_집단급식소  사업장폐기물배출자_음식점200이상  \
0            15.4           NaN              NaN                 NaN

In [34]:
df_master = pd.read_csv(file_master, encoding="utf-8-sig")

df_merged3 = df_master.merge(
    df_food,
    on=["자치구", "연도"],
    how="left"
)

df_merged3.to_csv(
    final_path / "생활폐기물_발생및처리_1인당_재활용_음식물발생원포함_master.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df_merged3.head())
print(df_merged3.info())

  대분류  자치구    연도    발생량  재활용_합계    재활용    음식물     소각     매립  기타  ...  \
0   계  강남구  2000  794.1   377.0  207.0  170.0   32.3  384.8 NaN  ...   
1   계  강남구  2001  799.0   510.7  242.1  268.6  158.7  129.6 NaN  ...   
2   계  강남구  2002  861.3   567.9  295.0  272.9  162.8  130.6 NaN  ...   
3   계  강남구  2003  874.9   556.8  267.3  289.5  169.5  148.6 NaN  ...   
4   계  강남구  2004  835.0   541.4  242.4  299.0  168.0  125.6 NaN  ...   

   다량배출사업장_음식점200이상  다량배출사업장_대규모점포  다량배출사업장_농수산시장 다량배출사업장_관광숙박시설  \
0               NaN            NaN            NaN            NaN   
1               NaN            NaN            NaN            NaN   
2               NaN            NaN            NaN            NaN   
3               NaN            NaN            NaN            NaN   
4               NaN            NaN            NaN            NaN   

   사업장폐기물배출자_소계  사업장폐기물배출자_집단급식소  사업장폐기물배출자_음식점200이상  사업장폐기물배출자_대규모점포  \
0           NaN              NaN                 NaN              NaN   
1           

In [35]:
check_cols = [
    "자치구", "연도",
    "음식물총량",
    "가정_소계",
    "다량배출사업장_소계",
    "사업장폐기물배출자_소계"
]

print(df_merged3[check_cols].head(15))
print(df_merged3["음식물총량"].isna().sum())
print(df_merged3["가정_소계"].isna().sum())
print(df_merged3["다량배출사업장_소계"].isna().sum())

    자치구    연도  음식물총량  가정_소계  다량배출사업장_소계  사업장폐기물배출자_소계
0   강남구  2000    NaN    NaN         NaN           NaN
1   강남구  2001    NaN    NaN         NaN           NaN
2   강남구  2002    NaN    NaN         NaN           NaN
3   강남구  2003    NaN    NaN         NaN           NaN
4   강남구  2004    NaN    NaN         NaN           NaN
5   강남구  2005    NaN    NaN         NaN           NaN
6   강남구  2006    NaN    NaN         NaN           NaN
7   강남구  2007    NaN    NaN         NaN           NaN
8   강남구  2008    NaN    NaN         NaN           NaN
9   강남구  2009    NaN    NaN         NaN           NaN
10  강남구  2010    NaN    NaN         NaN           NaN
11  강남구  2011    NaN    NaN         NaN           NaN
12  강남구  2012    NaN    NaN         NaN           NaN
13  강남구  2013    NaN    NaN         NaN           NaN
14  강남구  2014  342.3  190.5       151.8           NaN
375
375
375
